# Sesion 8 — De Datos Crudos a Dataset Analizable
## Diplomado: Machine Learning en Seguros · FC UNAM
### 2 de mayo de 2026  ·  07:00 - 11:00 h  (4 horas)

---

> **Premisa de la sesion:** recibes los archivos crudos de la aseguradora.
> Tu trabajo: convertirlos en un dataset limpio, bien tipado y eficiente,
> resolviendo 7 problemas reales uno por uno.

---

**Prerequisito:** Debe existir la carpeta `datos/` con los archivos CSV.

## Las 7 Dudas que Resolvemos Hoy

| # | Duda | Herramienta |
|---|------|-------------|
| 1 | 46 columnas — ¿cuales necesito? | Taxonomia + `usecols` |
| 2 | Texto sucio: M/F/Masculino/Femenino | `str` operations |
| 3 | Fechas como texto: '15/04/2026' | `pd.to_datetime()` |
| 4 | API de reaseguro devuelve JSON | `read_json()` + `json_normalize()` |
| 5 | 90k filas, 11MB sin esperar | `chunks` + `category` |
| 6 | Downcast rompio precision de primas | Estrategia segura de `dtypes` |
| 7 | ¿Cuando cambiar a Polars? | `polars` — intro y comparativa |

---
## ACT 1 — El Dataset Crudo

### Duda 1: 46 columnas — ¿cuales necesito?

In [1]:
import pandas as pd
import numpy as np
import os, time

# ── Paso 1: Cargar TODO para entender que hay ────────────────────────────────
# Primera regla: antes de descartar, entende lo que tienes
df_todo = pd.read_csv('datos/cartera_polizas.csv', nrows=5)  # solo 5 filas para ver
print(f'Columnas totales: {len(df_todo.columns)}')
print()
print('Lista de columnas:')
for i, col in enumerate(df_todo.columns, 1):
    print(f'  {i:>2}. {col}')

Columnas totales: 46

Lista de columnas:
   1. id_poliza
   2. num_poliza
   3. id_contrato_interno
   4. folio_emision
   5. id_sistema_legacy
   6. nombre
   7. apellido_paterno
   8. apellido_materno
   9. nombre_completo
  10. rfc
  11. fecha_nacimiento
  12. edad
  13. sexo
  14. estado_civil
  15. ocupacion
  16. nivel_educacion
  17. ramo
  18. plan
  19. fecha_emision
  20. fecha_inicio_vigencia
  21. fecha_fin_vigencia
  22. num_renovaciones
  23. status_poliza
  24. motivo_baja
  25. canal_venta
  26. marca_vehiculo
  27. modelo_vehiculo
  28. tipo_vehiculo
  29. suma_asegurada
  30. deducible
  31. prima_neta
  32. prima_total
  33. cuota_prima
  34. forma_pago
  35. num_cuotas
  36. agente_id
  37. estado
  38. municipio
  39. codigo_postal
  40. coord_lat
  41. coord_lon
  42. version_documento
  43. hash_documento
  44. timestamp_carga
  45. usuario_captura
  46. ip_carga


In [ ]:
# ── Paso 2: Diagnostico de todas las columnas ────────────────────────────────
# Cargamos todo para el diagnostico inicial
df_full = pd.read_csv('datos/cartera_polizas.csv')
mb_full = df_full.memory_usage(deep=True).sum() / 1024**2
print(f'Dataset completo: {df_full.shape} · {mb_full:.1f} MB')
print()

# Perfil de cada columna: tipo, NaN%, valores unicos
print(f'{"Columna":<30} {"Dtype":<12} {"NaN%":>7} {"Unicos":>8}')
print('-' * 65)
for col in df_full.columns:
    dtype = str(df_full[col].dtype)
    nan_pct = df_full[col].isna().mean() * 100
    n_uniq  = df_full[col].nunique()
    print(f'{col:<30} {dtype:<12} {nan_pct:>6.1f}% {n_uniq:>8,}')

In [ ]:
# ── Paso 3: Clasificar las columnas por categoria ────────────────────────────
# Esta clasificacion es una DECISION DE NEGOCIO — no solo tecnica

ANALITICAS = [
    'id_poliza','num_poliza','ramo','plan','status_poliza',
    'nombre','apellido_paterno','apellido_materno',
    'rfc','edad','sexo','estado_civil','ocupacion',
    'fecha_emision','fecha_inicio_vigencia','fecha_fin_vigencia',
    'num_renovaciones','motivo_baja',
    'suma_asegurada','deducible','prima_neta','prima_total',
    'forma_pago','agente_id','canal_venta',
    'estado','municipio','codigo_postal',
    'marca_vehiculo','modelo_vehiculo','tipo_vehiculo',  # solo Autos
]

REDUNDANTES = [
    'nombre_completo',        # = nombre + ap_pat + ap_mat
    'id_contrato_interno',    # ≈ id_poliza
    'folio_emision',          # ≈ num_poliza
    'cuota_prima',            # = prima_total / num_cuotas
    'num_cuotas',             # derivado de forma_pago
]

ADMINISTRATIVAS = [
    'hash_documento',         # hash SHA del PDF — auditoria IT
    'timestamp_carga',        # mismo valor para todos
    'ip_carga',               # IP del servidor batch
    'usuario_captura',        # operacion interna
    'version_documento',      # version del formato del contrato
    'id_sistema_legacy',      # util SOLO para joins con sistema core
]

CONDICIONALES = [
    'nivel_educacion',        # si lo necesitas para el modelo
    'coord_lat','coord_lon',  # solo si haces analisis geoespacial
]

print(f'Analiticas:     {len(ANALITICAS)}')
print(f'Redundantes:    {len(REDUNDANTES)}')
print(f'Administrativas:{len(ADMINISTRATIVAS)}')
print(f'Condicionales:  {len(CONDICIONALES)}')
print(f'Total:          {len(ANALITICAS)+len(REDUNDANTES)+len(ADMINISTRATIVAS)+len(CONDICIONALES)}')

In [ ]:
# ── Paso 4: Cargar SOLO las columnas que necesitamos ─────────────────────────
t0 = time.time()
df = pd.read_csv(
    'datos/cartera_polizas.csv',
    usecols=ANALITICAS,
    na_values=['N/D','N/A','ND','--','Sin dato',''],
)
t1 = time.time()
mb_opt = df.memory_usage(deep=True).sum() / 1024**2

print('=== COMPARATIVA DE CARGA ===')
print(f'Carga completa (46 cols):  {mb_full:.1f} MB')
print(f'Carga selectiva ({len(ANALITICAS)} cols):  {mb_opt:.1f} MB')
print(f'Reduccion de memoria:      {(1-mb_opt/mb_full)*100:.0f}%')
print(f'Tiempo de carga:           {(t1-t0)*1000:.0f} ms')
print()
print(f'Dataset de trabajo: {df.shape}')
df.head(3)

### 📝 Ejercicio 1 — Auditoria de columnas (8 min)

Usando el perfil que generaste arriba:
- **1a.** Identifica las 3 columnas con mas NaN — ¿tienen sentido esos NaN o son errores?
- **1b.** Encuentra columnas con menos de 5 valores unicos — ¿cuales deberian ser `category`?
- **1c.** Hay una columna numerica con un rango imposible (negativo o muy alto) — ¿cual es?
  *Pista: usa `df.describe()` sobre las columnas analiticas*

In [ ]:
# Tu codigo aqui:


---
### Duda 2: Texto Sucio — str Operations

El campo `sexo` tiene 4 representaciones distintas del mismo valor.
Si no lo normalizas, `groupby('sexo')` produce 8 grupos en lugar de 2.

In [ ]:
# ── Ver el problema ──────────────────────────────────────────────────────────
print('Valores unicos en sexo ANTES de limpiar:')
print(df['sexo'].value_counts(dropna=False))
print(f'Total valores distintos: {df["sexo"].nunique()}')
print()

# Si haces groupby ahora, obtienes grupos incorrectos:
grupos_incorrectos = df.groupby('sexo')['prima_total'].mean()
print(f'Grupos sin limpiar: {len(grupos_incorrectos)} (deberian ser 2)')

In [ ]:
# ── Solucion: normalizar con str operations ──────────────────────────────────

# Paso 1: normalizar a mayusculas sin espacios
df['sexo'] = df['sexo'].str.strip().str.upper()

# Paso 2: mapear todas las variantes al estandar
MAPA_SEXO = {
    'M': 'M', 'MASCULINO': 'M', 'HOMBRE': 'M', 'MASC': 'M',
    'F': 'F', 'FEMENINO': 'F', 'MUJER': 'F', 'FEM': 'F',
}
df['sexo'] = df['sexo'].map(MAPA_SEXO)
# Valores no reconocidos quedan como NaN automaticamente

print('DESPUES de normalizar:')
print(df['sexo'].value_counts(dropna=False))
print()
# Ahora groupby correcto
print('Prima promedio por sexo:')
print(df.groupby('sexo')['prima_total'].mean().round(2))

In [ ]:
# ── Mas str operations sobre los datos reales ────────────────────────────────

# Limpiar codigo_postal: eliminar 'N/D' (ya convertido a NaN por na_values)
# Verificar:
print(f'CP con NaN: {df["codigo_postal"].isna().sum()}')
# Rellenar CP desconocido con 'DESCONOCIDO' para no perder la fila
df['codigo_postal'] = df['codigo_postal'].fillna('DESCONOCIDO')

# Extraer ramo y anio desde num_poliza ('GMM-24-000123')
df['ramo_codigo']  = df['num_poliza'].str.extract(r'^([A-Z]+)-')
df['anio_poliza']  = df['num_poliza'].str.extract(r'-([0-9]{2})-').astype(float).astype('Int16')

# Verificar
print(df[['num_poliza','ramo_codigo','anio_poliza']].head(5).to_string(index=False))

---
### Duda 3: Fechas como Texto — pd.to_datetime()

In [ ]:
# ── El problema: fechas llegaron como strings ────────────────────────────────
print('Tipo ANTES de convertir:', df['fecha_nacimiento'].dtype)
print('Muestra:', df['fecha_nacimiento'].head(3).values)
print()

# ── Convertir fecha_nacimiento (formato d/m/Y) ────────────────────────────────
df['fecha_nacimiento'] = pd.to_datetime(
    df['fecha_nacimiento'],
    format='%d/%m/%Y',
    errors='coerce'  # fechas invalidas → NaT (no detiene el proceso)
)

# ── Convertir fechas ISO estandar ────────────────────────────────────────────
for col_fecha in ['fecha_emision','fecha_inicio_vigencia','fecha_fin_vigencia']:
    df[col_fecha] = pd.to_datetime(df[col_fecha], errors='coerce')

print('Fechas convertidas:')
print(df[['fecha_nacimiento','fecha_emision','fecha_fin_vigencia']].dtypes)
print(f'NaT en fecha_nacimiento: {df["fecha_nacimiento"].isna().sum()}')

In [ ]:
# ── Calculos actuariales con las fechas convertidas ──────────────────────────
hoy = pd.Timestamp.today()

# Edad calculada (mas precisa que la columna 'edad' del CSV)
df['edad_calc'] = ((hoy - df['fecha_nacimiento']).dt.days / 365.25).astype('Int8')

# Dias de vigencia de la poliza
df['dias_vigencia'] = (df['fecha_fin_vigencia'] - df['fecha_inicio_vigencia']).dt.days

# Fraccion expuesta (cuanto del periodo ya transcurrio)
dias_transcurridos = (hoy - df['fecha_inicio_vigencia']).dt.days
df['fraccion_expuesta'] = (dias_transcurridos / df['dias_vigencia']).clip(0, 1).round(4)

# Componentes de fecha para groupby temporal
df['anio_emision']     = df['fecha_emision'].dt.year
df['mes_emision']      = df['fecha_emision'].dt.month
df['trimestre_emision']= df['fecha_emision'].dt.quarter

# Verificar
print(df[['nombre','edad','edad_calc','dias_vigencia','fraccion_expuesta']].head(5).to_string(index=False))

### 📝 Ejercicio 2 — Limpiar siniestros.csv (10 min)

El archivo `siniestros.csv` tiene fechas en 3 formatos distintos:
- `fecha_ocurrencia`: YYYY-MM-DD
- `fecha_apertura`: d/m/Y (mismo dia que fecha_reporte pero formato distinto — es REDUNDANTE)
- `fecha_ultimo_movimiento`: d/m/Y

**2a.** Carga siniestros.csv usando SOLO las columnas utiles (descarta las administrativas y redundantes).
**2b.** Convierte las 3 columnas de fecha a datetime con el formato correcto.
**2c.** Calcula `dias_reporte_real` = fecha_reporte - fecha_ocurrencia.
**2d.** Calcula `dias_resolucion_real` = fecha_cierre - fecha_reporte (NaT si no esta cerrado).
**2e.** Verifica: ¿cuantos siniestros llevan mas de 180 dias sin cerrar?

In [ ]:
# Tu codigo aqui:
siniestros_cols_utiles = [
    'id_siniestro','id_poliza','ramo','tipo_siniestro',
    'fecha_ocurrencia','fecha_reporte','fecha_ultimo_movimiento','fecha_cierre',
    'dias_reporte','monto_reclamado','monto_pagado',
    'status_siniestro','motivo_rechazo','id_ajustador',
]
# Carga, convierte fechas y calcula los campos derivados:


---
### Duda 4: JSON — Datos de API


In [ ]:
# ── Simular una respuesta JSON de una API de reaseguro ───────────────────────
import json

# Este es el tipo de JSON que recibirias de una API REST
respuesta_api = {
    'timestamp': '2026-05-02T07:00:00',
    'origen': 'Sistema_Reaseguro_v3.1',
    'polizas_reaseguradas': [
        {'id_poliza':'POL-000001','ramo':'GMM',
         'reasegurador':{'nombre':'Munich Re','participacion':0.40,'prima':960.0},
         'limites':{'maximo':2_000_000,'retencion':500_000}},
        {'id_poliza':'POL-000002','ramo':'Vida',
         'reasegurador':{'nombre':'Swiss Re','participacion':0.35,'prima':1820.0},
         'limites':{'maximo':5_000_000,'retencion':1_000_000}},
        {'id_poliza':'POL-000005','ramo':'GMM',
         'reasegurador':{'nombre':'Munich Re','participacion':0.40,'prima':550.0},
         'limites':{'maximo':2_000_000,'retencion':500_000}},
    ]
}

# Guardar como JSON (simula lo que llegaria de la API)
with open('datos/respuesta_reaseguro.json','w',encoding='utf-8') as f:
    json.dump(respuesta_api, f, ensure_ascii=False, indent=2)

print('JSON guardado en datos/respuesta_reaseguro.json')
print('Primeras lineas:')
print(json.dumps(respuesta_api, indent=2, ensure_ascii=False)[:300])

In [ ]:
# ── Problema: JSON anidado no se carga directo en DataFrame ─────────────────
# pd.read_json funciona para JSON simple pero no para estructuras anidadas

# Cargar el JSON
with open('datos/respuesta_reaseguro.json') as f:
    data = json.load(f)

# Intentar con pd.read_json — no da el resultado esperado con anidados
# pd.read_json('datos/respuesta_reaseguro.json')  # solo lee nivel 1

# ── Solucion: json_normalize aplana el JSON anidado ──────────────────────────
from pandas import json_normalize

df_reas = json_normalize(
    data['polizas_reaseguradas'],
    sep='_'    # separador para campos anidados
)

print('DataFrame aplanado:')
print(df_reas.to_string(index=False))
print()
print('Columnas generadas:', list(df_reas.columns))

In [ ]:
# ── json_normalize con listas anidadas ───────────────────────────────────────
# Caso mas complejo: cuando hay listas dentro del JSON

respuesta_multi = {
    'polizas': [
        {'id':'P01','coberturas':[{'tipo':'Hospitalizacion','suma':500_000},{'tipo':'Cirugia','suma':300_000}]},
        {'id':'P02','coberturas':[{'tipo':'Hospitalizacion','suma':800_000}]},
    ]
}

# record_path: donde esta la lista a 'explotar'
# meta: campos del nivel padre que quieres conservar
df_cob = json_normalize(
    respuesta_multi['polizas'],
    record_path='coberturas',
    meta=['id'],
    sep='_'
)
print(df_cob)

In [ ]:
# ── Guardar DataFrame como JSON ──────────────────────────────────────────────

# orient='records' — lista de objetos (lo mas comun para APIs)
df.head(10).to_json('datos/muestra.json',
    orient='records',
    force_ascii=False,  # preserva caracteres especiales (acentos)
    indent=2,
    date_format='iso'   # fechas en formato ISO
)

# Verificar
with open('datos/muestra.json') as f:
    preview = f.read(300)
print(preview)

---
### Duda 5: 90k Filas — Procesar sin Esperar (chunks)

In [ ]:
# ── Ver el tamano del archivo de beneficiarios ───────────────────────────────
mb_ben = os.path.getsize('datos/beneficiarios.csv') / 1024**2
print(f'beneficiarios.csv: {mb_ben:.1f} MB')

# Contar filas sin cargar todo
total_ben = sum(len(chunk) for chunk in
    pd.read_csv('datos/beneficiarios.csv', chunksize=10_000))
print(f'Total beneficiarios: {total_ben:,}')

# ── Patron real: calcular estadisticas por parentesco ─────────────────────────
# Sin cargar los 90k en memoria a la vez
conteo_parentesco = {}

for chunk in pd.read_csv('datos/beneficiarios.csv', chunksize=10_000):
    counts = chunk['parentesco'].value_counts().to_dict()
    for k, v in counts.items():
        conteo_parentesco[k] = conteo_parentesco.get(k, 0) + v

resultado = pd.Series(conteo_parentesco).sort_values(ascending=False)
print('Beneficiarios por parentesco (procesado por chunks):')
print(resultado)

In [ ]:
# ── Filtrar y concatenar solo lo que necesitas ───────────────────────────────
# Obtener solo beneficiarios activos de polizas de Vida

partes = []
for chunk in pd.read_csv('datos/beneficiarios.csv',
                         chunksize=10_000,
                         usecols=['id_beneficiario','id_poliza','nombre',
                                  'apellido_paterno','parentesco','porcentaje','activo']):
    filtrado = chunk[chunk['activo'] == True]
    if len(filtrado) > 0:
        partes.append(filtrado)

ben_activos = pd.concat(partes, ignore_index=True)
print(f'Beneficiarios activos: {len(ben_activos):,}')
print(f'Memoria: {ben_activos.memory_usage(deep=True).sum()/1024:.0f} KB')

---
### Duda 6: Downcast — Cuándo Es Seguro

In [ ]:
# ── Demostrar el problema de precision ───────────────────────────────────────
import numpy as np

# float64 vs float32 con valores grandes
prima_grande = 15_432_756.80
print(f'Original (float64): {prima_grande}')
print(f'Como float32:       {np.float32(prima_grande)}')
print(f'Diferencia:         {abs(prima_grande - float(np.float32(prima_grande))):.2f}')
print()

# Con valores tipicos de primas individuales
prima_gmm = 3_450.75
print(f'Prima GMM (float64): {prima_gmm}')
print(f'Prima GMM (float32): {np.float32(prima_gmm)}')
print(f'Diferencia:          {abs(prima_gmm - float(np.float32(prima_gmm))):.6f}')

In [ ]:
# ── Estrategia segura de optimizacion ────────────────────────────────────────
print('Memoria ANTES de optimizar:')
print(f'{df.memory_usage(deep=True).sum()/1024**2:.2f} MB')
print()

df_opt = df.copy()

# SEGURO: categoricas para columnas con pocos valores unicos
cols_category = ['ramo','plan','status_poliza','sexo','canal_venta',
                 'forma_pago','estado','estado_civil','tipo_vehiculo']
for col in cols_category:
    if col in df_opt.columns:
        n_uniq = df_opt[col].nunique()
        n_tot  = len(df_opt)
        pct = n_uniq/n_tot
        print(f'  {col:<25}: {n_uniq:>5} unicos ({pct:.1%}) → category')
        df_opt[col] = df_opt[col].astype('category')

# SEGURO: enteros pequenos
df_opt['num_renovaciones'] = df_opt['num_renovaciones'].fillna(0).astype('int8')

# SEGURO: booleano
# (activa ya no esta porque la descartamos, pero aplica el principio)

# CONSERVAR en float64: primas y sumas (montos grandes o con centavos importantes)
# NO hacer: df_opt['prima_total'] = df_opt['prima_total'].astype('float32')

print()
print('Memoria DESPUES de optimizar:')
mb_antes = df.memory_usage(deep=True).sum()/1024**2
mb_desp  = df_opt.memory_usage(deep=True).sum()/1024**2
print(f'{mb_antes:.2f} MB → {mb_desp:.2f} MB ({(1-mb_desp/mb_antes)*100:.0f}% reduccion)')

### 📝 Ejercicio 3 — Pipeline de limpieza completo (12 min)

Encapsula todo lo que hicimos en una funcion `limpiar_cartera(ruta_csv)` que:
- Carga con `usecols=ANALITICAS`
- Normaliza `sexo` con str + map
- Convierte fechas con `to_datetime` + `errors='coerce'`
- Calcula `edad_calc`, `dias_vigencia`, `fraccion_expuesta`
- Aplica optimizacion de memoria (categoricas)
- Retorna el DataFrame limpio

Al final llama: `df_limpio = limpiar_cartera('datos/cartera_polizas.csv')`
y verifica que no tenga texto sucio en sexo.

In [ ]:
# Tu codigo aqui:
def limpiar_cartera(ruta_csv):
    # Implementa la funcion completa aqui
    pass


---
### Duda 7: ¿Cuándo Cambiar a Polars?

In [ ]:
# ── Instalar polars si no esta disponible ────────────────────────────────────
# En tu terminal: pip install polars
# Verificar:
try:
    import polars as pl
    print(f'Polars {pl.__version__} disponible')
    POLARS_OK = True
except ImportError:
    print('Polars no instalado. Ejecuta: pip install polars')
    POLARS_OK = False

In [ ]:
# ── Comparativa de velocidad pandas vs polars ────────────────────────────────
if POLARS_OK:
    import polars as pl
    import time

    # ── Pandas ──────────────────────────────────────────────────────────────
    t0 = time.time()
    df_pd = pd.read_csv('datos/cartera_polizas.csv', usecols=ANALITICAS)
    res_pd = (df_pd.groupby('ramo')
                   .agg(polizas=('id_poliza','count'),
                        prima_total=('prima_total','sum'),
                        prima_prom=('prima_neta','mean'))
                   .round(2).reset_index())
    t_pd = time.time()-t0

    # ── Polars ──────────────────────────────────────────────────────────────
    t0 = time.time()
    df_pl = pl.read_csv('datos/cartera_polizas.csv', columns=ANALITICAS)
    res_pl = (df_pl
        .group_by('ramo')
        .agg([
            pl.col('id_poliza').count().alias('polizas'),
            pl.col('prima_total').sum().alias('prima_total'),
            pl.col('prima_neta').mean().alias('prima_prom'),
        ])
    )
    t_pl = time.time()-t0

    print(f'Pandas:  {t_pd*1000:.0f} ms')
    print(f'Polars:  {t_pl*1000:.0f} ms')
    print(f'Polars es {t_pd/t_pl:.1f}x mas rapido en esta operacion')
    print()
    print('Resultado Polars:')
    print(res_pl)
else:
    print('Instala polars para ver la comparativa: pip install polars')

In [ ]:
# ── Sintaxis Polars: las operaciones mas comunes ──────────────────────────────
if POLARS_OK:
    df_pl = pl.read_csv('datos/cartera_polizas.csv',
                        columns=['id_poliza','ramo','prima_total','siniest','zona','edad'])

    # Filtrar
    print('Polizas con prima > 10,000:')
    print(df_pl.filter(pl.col('prima_total') > 10_000).shape)

    # Agregar columna
    df_pl2 = df_pl.with_columns(
        (pl.col('prima_total')/12).alias('prima_mensual'),
        (pl.col('siniest') >= 2).alias('alto_riesgo'),
    )

    # Sort
    print('Top 5 primas:')
    print(df_pl2.sort('prima_total', descending=True).head(5)[['id_poliza','ramo','prima_total']])

    # Lazy evaluation — ejecuta todo optimizado al final
    resultado = (
        pl.scan_csv('datos/cartera_polizas.csv')  # NO carga en memoria aun
        .filter(pl.col('prima_total') > 5000)
        .group_by('ramo')
        .agg(pl.col('prima_total').sum())
        .collect()  # AHORA ejecuta con el plan optimizado
    )
    print('Lazy evaluation result:')
    print(resultado)

### Regla practica: ¿Pandas o Polars?

| Situacion | Usa |
|-----------|-----|
| Aprendizaje, primeros modelos | **pandas** — el ecosistema es enorme |
| < 1 millon de filas | **pandas** — mas que suficiente |
| Analisis interactivo en notebook | **pandas** — syntax mas conocida |
| Pipeline de produccion > 5M filas | **polars** — 5-20x mas rapido |
| Ingesta diaria de datos grandes | **polars** con lazy evaluation |
| ETL de empresa | **polars** o **spark** segun el tamano |

---
## pivot_table con Datos Reales


In [ ]:
# ── Agregar columnas necesarias para el pivot ────────────────────────────────
df_work = pd.read_csv('datos/cartera_polizas.csv', usecols=ANALITICAS,
                      na_values=['N/D','N/A',''])
df_work['prima_neta'] = df_work['prima_neta'].fillna(df_work.groupby('ramo')['prima_neta'].transform('median'))
df_work['g_edad'] = pd.cut(df_work['edad'], bins=[0,30,45,60,100],
                           labels=['18-30','31-45','46-60','61+'])
df_work['siniest_flag'] = (df_work['prima_neta'] > df_work['prima_neta'].quantile(0.75)).astype(int)

print(f'Dataset para pivot: {df_work.shape}')

In [ ]:
# ── pivot_table: prima por ramo x grupo de edad ──────────────────────────────
tabla_prima = pd.pivot_table(
    df_work,
    values   = 'prima_total',
    index    = 'ramo',
    columns  = 'g_edad',
    aggfunc  = 'sum',
    fill_value = 0,
    margins    = True,
    margins_name = 'TOTAL'
).round(0) / 1_000  # en miles de pesos

print('Prima total por ramo y grupo de edad (miles MXN):')
print(tabla_prima.to_string())

In [ ]:
# ── pivot_table: polizas por zona x canal ────────────────────────────────────
tabla_canal = pd.pivot_table(
    df_work,
    values   = 'id_poliza',
    index    = 'estado',
    columns  = 'canal_venta',
    aggfunc  = 'count',
    fill_value = 0,
    margins    = True,
    margins_name = 'TOTAL'
)

print('Polizas por estado y canal de venta:')
print(tabla_canal.to_string())

---
## Exportar — CSV, Excel, Parquet y JSON


In [ ]:
# ── Guardar en todos los formatos y comparar ──────────────────────────────────
df_export = df_work.head(10_000)  # subconjunto para demo

formatos = {
    'CSV':     ('datos/export_demo.csv',
                lambda: df_export.to_csv('datos/export_demo.csv', index=False)),
    'Parquet': ('datos/export_demo.parquet',
                lambda: df_export.to_parquet('datos/export_demo.parquet', index=False)),
    'JSON':    ('datos/export_demo.json',
                lambda: df_export.to_json('datos/export_demo.json',
                                          orient='records', force_ascii=False)),
}

print(f'{"Formato":<10} {"Tamano":>10} {"Tiempo":>10}')
print('-' * 35)
for nombre, (ruta, guardar) in formatos.items():
    t0 = time.time()
    guardar()
    t = (time.time()-t0)*1000
    kb = os.path.getsize(ruta)/1024
    print(f'{nombre:<10} {kb:>8.0f} KB {t:>8.0f} ms')

In [ ]:
# ── Excel multihoja — el entregable mas solicitado en empresas ───────────────
resumen_ramo = df_work.groupby('ramo').agg(
    polizas    =('id_poliza','count'),
    prima_total=('prima_total','sum'),
    prima_prom =('prima_total','mean'),
).round(2).reset_index()
resumen_ramo['pct_cartera'] = (resumen_ramo['prima_total']/resumen_ramo['prima_total'].sum()*100).round(1)

with pd.ExcelWriter('datos/reporte_demo.xlsx', engine='openpyxl') as writer:
    df_work.head(5_000).to_excel(writer, sheet_name='Cartera', index=False)
    resumen_ramo.to_excel(writer, sheet_name='Resumen_Ramo', index=False)
    tabla_prima.to_excel(writer, sheet_name='Pivot_Prima')
    tabla_canal.to_excel(writer, sheet_name='Pivot_Canal')

kb_xl = os.path.getsize('datos/reporte_demo.xlsx')/1024
print(f'Excel multihoja generado: {kb_xl:.0f} KB con 4 hojas')

---
## 🏆 Ejercicio Integrador Final — Pipeline Completo

**Contexto:** Tu jefa de estadistica te pide el reporte ejecutivo del Q1 2026.
Tienes los 4 archivos crudos. Debes construir un pipeline completo,
documentando cada decision de limpieza.

**Tiempo:** 40 minutos  |  **Entregable:** Excel con 5 hojas + Parquet

---

### Criterios de evaluacion:
- ✅ Cada decision de descarte de columnas esta justificada con comentario
- ✅ No hay texto sucio en columnas categoricas clave
- ✅ Todas las fechas son datetime, no object
- ✅ dtypes optimizados sin perder precision en montos
- ✅ El Excel tiene las 5 hojas con los datos correctos
- ✅ El Parquet es mas pequeno que el CSV equivalente

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# FASE 1: INGESTA INTELIGENTE
# ══════════════════════════════════════════════════════════════════════════════
# Carga cartera, siniestros y catalogo de ramos/agentes.
# Usa SOLO las columnas que necesitas — justifica con comentario.
# Mide la memoria ahorrada vs cargar todo.

# Tu codigo aqui:


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# FASE 2: LIMPIEZA COMPLETA
# ══════════════════════════════════════════════════════════════════════════════
# 2a. Elimina duplicados de cartera
# 2b. Normaliza sexo con str.strip().str.upper() + .map(MAPA_SEXO)
# 2c. Convierte TODAS las fechas a datetime con errors='coerce'
# 2d. Rellena NaN de prima_neta con la MEDIANA POR RAMO (no global)
#     df.groupby('ramo')['prima_neta'].transform(lambda x: x.fillna(x.median()))
# 2e. Limpia codigo_postal (reemplaza 'N/D' con NaN)
# 2f. Aplica optimizacion de categoricas (sin tocar float64 de primas)

# Tu codigo aqui:


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# FASE 3: ENRIQUECIMIENTO
# ══════════════════════════════════════════════════════════════════════════════
# 3a. Merge con catalogo_ramos: agregar nombre_largo, tasa_base
# 3b. Merge con catalogo_agentes: agregar nombre del agente, region
# 3c. Crear: g_edad con pd.cut
# 3d. Crear: prima_calc = suma_asegurada * tasa_base * 1.16
# 3e. Crear: nivel_riesgo con .apply(clasificar_riesgo) — de mi_modulo
# 3f. Crear: edad_calc desde fecha_nacimiento
# 3g. Crear: dias_vigencia, fraccion_expuesta

# Tu codigo aqui:


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# FASE 4: ANALISIS Y REPORTES
# ══════════════════════════════════════════════════════════════════════════════
# 4a. groupby+agg por ramo: polizas, prima_total, prima_prom, pct_cartera
# 4b. groupby+agg por agente: polizas, prima_total, comision (10%)
# 4c. pivot_table prima por ramo x g_edad con margins=True
# 4d. pivot_table polizas por estado x ramo
# 4e. Identifica: ramo con mayor prima total y zona con mayor frecuencia

# Tu codigo aqui:


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# FASE 5: EXPORTAR
# ══════════════════════════════════════════════════════════════════════════════
# Excel con 5 hojas: Cartera_Limpia, Resumen_Ramo, Resumen_Agente,
#                    Pivot_Prima, Pivot_Zona
# Parquet: cartera_q1_2026_final.parquet
# Compara tamano CSV equivalente vs Parquet

# Tu codigo aqui:


In [ ]:
# ── Solucion resumida (descomenta solo si necesitas referencia) ──────────────

# FASE 1:
# df = pd.read_csv('datos/cartera_polizas.csv', usecols=ANALITICAS, na_values=['N/D','N/A'])
# ramos_cat = pd.read_csv('datos/catalogo_ramos.csv')
# agentes_cat = pd.read_csv('datos/catalogo_agentes.csv')

# FASE 2:
# df = df.drop_duplicates()
# df['sexo'] = df['sexo'].str.strip().str.upper().map(MAPA_SEXO)
# for col in ['fecha_emision','fecha_inicio_vigencia','fecha_fin_vigencia']:
#     df[col] = pd.to_datetime(df[col], errors='coerce')
# df['fecha_nacimiento'] = pd.to_datetime(df['fecha_nacimiento'],format='%d/%m/%Y',errors='coerce')
# df['prima_neta'] = df.groupby('ramo')['prima_neta'].transform(lambda x: x.fillna(x.median()))
# df['codigo_postal'] = df['codigo_postal'].replace('N/D', np.nan)
# for col in ['ramo','plan','canal_venta','forma_pago','estado','sexo','tipo_vehiculo']:
#     if col in df.columns: df[col] = df[col].astype('category')

# FASE 3 (extracto):
# df = pd.merge(df, ramos_cat[['ramo','nombre_largo','tasa_base']], on='ramo', how='left')
# df['g_edad'] = pd.cut(df['edad'],bins=[0,30,45,60,100],labels=['18-30','31-45','46-60','61+'])
# df['prima_calc'] = df['suma_asegurada'] * df['tasa_base'] * 1.16
# from mi_modulo import clasificar_riesgo
# df['nivel_riesgo'] = df['prima_calc'].apply(lambda p: 'ALTO' if p>15000 else 'MEDIO' if p>6000 else 'BAJO')

# FASE 5:
# with pd.ExcelWriter('datos/reporte_ejecutivo_Q1_2026.xlsx', engine='openpyxl') as w:
#     df.to_excel(w,'Cartera_Limpia',index=False)
#     resumen_ramo.to_excel(w,'Resumen_Ramo',index=False)
#     resumen_agente.to_excel(w,'Resumen_Agente',index=False)
#     tabla_prima.to_excel(w,'Pivot_Prima')
#     tabla_zona.to_excel(w,'Pivot_Zona')
# df.to_parquet('datos/cartera_q1_2026_final.parquet', index=False)
# print(f'CSV: {df.to_csv(index=False).encode().__len__()/1024:.0f} KB')
# print(f'Parquet: {os.path.getsize("datos/cartera_q1_2026_final.parquet")/1024:.0f} KB')

---
## Resumen: Lo que Aprendiste en la Sesion 8

| Duda | Herramienta | Aprendizaje clave |
|------|-------------|-------------------|
| 46 columnas | `usecols` + taxonomia | Clasificar antes de cargar — decision de negocio |
| Texto sucio | `str.strip/upper/map` | Normalizar ANTES del primer groupby |
| Fechas texto | `pd.to_datetime(errors='coerce')` | Nunca detener el pipeline por fechas invalidas |
| JSON anidado | `json_normalize(sep='_')` | Aplanar antes de analizar |
| 90k filas | `chunksize` + `category` | Medir memoria antes de decidir |
| Downcast riesgoso | Regla float32 < 100k MXN | float64 para montos grandes siempre |
| Polars | `pl.scan_csv().collect()` | Lazy evaluation = optimizacion automatica |

**T5 Pandas — COMPLETADO**

**Proxima sesion — Mie 6 mayo, 18:00 h:**
T6 Visualizacion — Matplotlib, Seaborn y Plotly.

```bash
git add sesion8_M1_notebook.ipynb
git commit -m "Sesion 8: pipeline completo datos reales - str datetime JSON Polars"
git push
```

---
*Diplomado ML en Seguros · FC UNAM · 2026*